# Lesson 24 · One chain of multiplications holds up all of deep learning — the chain rule, the seed of backpropagation

**Learning goals**: the chain rule for composite functions, $\frac{dy}{dx}=\frac{dy}{du}\cdot\frac{du}{dx}$ — multiply layer by layer. Work it out by hand first on a minimal 2–3-node example; this is the algebraic core of backpropagation (L54–L56 build it into the Value engine).

**Why it matters for Aurora**: autograd's `backward()` multiplies local derivatives along the computation graph — this lesson's formula is each link in that chain.

← **Previous lesson** · [L23 · Gradients](L23_gradients.ipynb)

> Last lesson covered **gradients**: the "steepest uphill" direction of a multivariable function, and computing partial derivatives and the gradient vector.
> This lesson explores **the chain rule**.

## This lesson's story: what does backpropagation run on?

A neural network has dozens of layers. If you nudge the input a tiny bit, how much does the final loss change?

The answer is the chain rule — the rule for differentiating a function nested inside a function:

```
y = f(g(x))  →  dy/dx = f'(g(x)) · g'(x)
```

Intuition: the inner layer amplifies (or shrinks) a wiggle in x by a factor of g'(x), and the outer layer then amplifies the wiggle in the intermediate value by a factor of f'(g). Multiply the two amplification factors together and you get the total amplification.

**Example**: y = sin(x²)
- Outer layer f(u) = sin(u), f'(u) = cos(u)
- Inner layer g(x) = x², g'(x) = 2x
- dy/dx = cos(x²) · 2x

At x=1: dy/dx = cos(1) · 2 ≈ 1.080

Backpropagation (backprop, BP) is the chain rule carried out in bulk on a computation graph:
starting from the error at the last layer, it "passes the product of derivatives" back one layer at a time, until the gradient of every parameter has been computed.

In this lesson you'll implement `composite_derivative(x)` and check it with a numerical difference, paving the way for backpropagation in L56.

## Background top-up: three important ideas in this lesson

Before diving into the chain rule, let's sort out three ideas that are easy to mix up.

### 1. Why the central difference? Won't a one-sided difference do?

In high-school maths, the derivative is defined as $\lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$ (one-sided), approaching from the right.
In numerical computation, we use the central difference $\frac{f(x+h) - f(x-h)}{2h}$ — approaching from both sides at once.

**Why is the central difference more accurate?** Use a Taylor expansion:
- $f(x+h) = f(x) + f'(x)h + \frac{f''(x)}{2}h^2 + \frac{f'''(x)}{6}h^3 + O(h^4)$
- $f(x-h) = f(x) - f'(x)h + \frac{f''(x)}{2}h^2 - \frac{f'''(x)}{6}h^3 + O(h^4)$

Subtract the two: $f(x+h) - f(x-h) = 2f'(x)h + \frac{f'''(x)}{3}h^3 + O(h^5)$

So $\frac{f(x+h) - f(x-h)}{2h} = f'(x) + \frac{f'''(x)}{6}h^2 + O(h^4)$

The key: **the error term is $O(h^2)$, while the one-sided difference's error is $O(h)$**. So at step size $h=10^{-5}$ the central difference's error is about $10^{-10}$, while the one-sided difference only gets down to $10^{-5}$.

### 2. What's the difference between the partial symbol ∂ and the derivative symbol d?

For the single-variable function $y=\sin(x^2)$, why do we sometimes write $\frac{dy}{dx}$ and sometimes $\frac{\partial y}{\partial x}$?

- $\frac{dy}{dx}$: an "ordinary derivative" — the function has only one input, so differentiating is unambiguous
- $\frac{\partial y}{\partial x}$: a "partial derivative" — the function may have several inputs (like the many weights of a neural network); $\partial$ stresses "change only $x$, hold the other variables fixed"

For a single-variable function, the two are exactly equivalent. **We use ∂ here to build the habit — the neural networks in Month 2 will have hundreds of weights, and each weight's "partial derivative" is its gradient.**

### 3. What exactly is a "local derivative"?

In the composite function $y=f(g(x))$:

| Term | Meaning | Example (y=sin(x²)) |
|---|---|---|
| **Global derivative** | the derivative from the final output $y$ back to the original input $x$ | $\frac{dy}{dx} = \cos(x^2) \cdot 2x$ |
| **Local derivative (derivative of an edge)** | the derivative along one edge of the computation graph (look only at that edge, ignore everything else) | edge $x \to u$: $\frac{du}{dx}=2x$; edge $u \to y$: $\frac{dy}{du}=\cos(u)$ |

Backpropagation splits the "global derivative" into a product of "local derivatives": $\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$ (that's the chain rule).

---

## 1. The chain rule: peel the layers apart, multiply layer by layer

**Why is it called a "chain"?**

Write the composite function as a chain:

```
x  →  g(x)=x²  →  f(u)=sin(u)  →  y=sin(x²)
```

The derivative travels along the chain, and at each step gets multiplied by that layer's "local slope":

$$\frac{dy}{dx} = \frac{df}{du} \cdot \frac{dg}{dx} = f'(g(x)) \cdot g'(x)$$

**Steps by hand** (y = sin(x²)):

| Step | Meaning | Result |
|---|---|---|
| Split the layers | let u = g(x) = x², y = f(u) = sin(u) | — |
| Inner derivative | du/dx = g'(x) = 2x | 2x |
| Outer derivative | dy/du = f'(u) = cos(u) = cos(x²) | cos(x²) |
| Chain product | dy/dx = cos(x²) · 2x | cos(x²)·2x |

**Numerical check (x=1.3)**: analytical value = cos(1.69)·2.6 ≈ -0.309; the central-difference result should agree to within 1e-9.

> **Generalizing to n layers**: every extra layer adds one more term to the chain of multiplications. Backpropagation (backprop) writes this chain of multiplications as matrix–vector products and passes gradients along in bulk.

**Note: why does f'(u) = cos(u) in the table end up as cos(x²)?**

This is the spot beginners most often mix up. Let's look at it from two viewpoints:

| Viewpoint | Process | Meaning |
|---|---|---|
| **Derivative with respect to u** | $f(u) = \sin(u)$, so $\frac{df}{du} = \cos(u)$ | this is the outer function's derivative, a function of u |
| **Back to an expression in x** | since $u = x^2$, swap $u$ back for $x^2$: $f'(u) = \cos(u) = \cos(x^2)$ | now the derivative is a function of x |

In other words:
- $f'(u)$ is the "general" form of the outer derivative — it holds for any value of u
- $f'(g(x)) = \cos(x^2)$ is the form "specialized" to our particular composite function — now it's written in terms of x

The two expressions don't contradict each other; they're just two ways of writing the same derivative. In code, we compute `np.cos(x**2)` directly, which does this substitution automatically.

---

### A rigorous derivation: why multiply and not add?

You're right to be suspicious — why isn't it $\frac{dy}{dx} = \frac{dy}{du} + \frac{du}{dx}$?

Derive it from the limit definition:

$$\frac{dy}{dx} = \lim_{h \to 0} \frac{f(g(x+h)) - f(g(x))}{h}$$

Let $\Delta u = g(x+h) - g(x)$, so that $g(x+h) = g(x) + \Delta u$.

$$\frac{dy}{dx} = \lim_{h \to 0} \frac{f(g(x) + \Delta u) - f(g(x))}{h}$$

Multiply by $\frac{\Delta u}{\Delta u}$ (when $\Delta u \ne 0$):

$$\frac{dy}{dx} = \lim_{h \to 0} \frac{f(g(x) + \Delta u) - f(g(x))}{\Delta u} \cdot \frac{\Delta u}{h}$$

In the first factor, $\Delta u \to 0$ as $h \to 0$, so $\lim_{h \to 0} \frac{f(g(x) + \Delta u) - f(g(x))}{\Delta u} = f'(g(x))$;

the second factor is $\lim_{h \to 0} \frac{\Delta u}{h} = \lim_{h \to 0} \frac{g(x+h) - g(x)}{h} = g'(x)$.

Therefore: $$\frac{dy}{dx} = f'(g(x)) \cdot g'(x)$$

**Key intuition**: a tiny change $\Delta u$ in the inner layer gets "amplified" by the outer layer (if $f'(u)$ is large) or "shrunk" (if $f'(u)$ is small) — that's a multiplication, not an addition. If it were an addition, it would mean "however the inner layer changes, the outer layer contributes a fixed amount", which doesn't hold mathematically.

---

## Into the lab: watching a function through numerical change

Below we compute the numerical derivative of `y = sin(x²)` at `x = 1.3` and compare it with the analytical value `cos(x²)·2x`. Pay attention to how big the gap between the `numerical` and `analytical` results is.

In [ ]:
import numpy as np
x = 1.3
analytic = np.cos(x**2) * 2*x          # chain rule
numeric = (np.sin((x+1e-5)**2) - np.sin((x-1e-5)**2)) / (2e-5)
print('chain rule:', round(analytic,5), ' numerical check:', round(numeric,5))

> **[Warm-up cells — not the chain rule]** The next few cells practice the "numerical difference" technique itself on the simplest functions (`f(x)=sin(x)`, `f(x)=x²`, `f(x)=x²+2x`).
> There's no nesting of functions involved, so they are **not applications of the chain rule**. The point is to confirm how accurate the difference formula is before you meet composite functions.
> The real chain-rule examples start in the next section (2. Implement `composite_derivative`).

## Hands-on: a rate of change is more than a slogan

Print the function values and the approximate slopes side by side over a set of `x` values, and see how accurate the numerical difference is at each point.

In [ ]:
import numpy as np

# Example: f(x) = sin(x); its derivative at x=1 is cos(1) ≈ 0.5403
# Approximate it with the two difference formulas and watch how the error changes
f = lambda x: np.sin(x)
true_derivative = np.cos(1.0)

print("Comparing the accuracy of the central vs one-sided difference for different step sizes h")
print(f"true derivative cos(1) = {true_derivative:.8f}\n")
print(f"{'h':>10}  {'one-sided error':>16}  {'central error':>16}  {'improvement':>12}")
print("-" * 60)

for h_power in range(-3, -12, -1):  # h = 1e-3, 1e-4, ..., 1e-11
    h = 10.0 ** h_power

    # one-sided difference: [f(x+h) - f(x)] / h
    one_sided = (f(1.0 + h) - f(1.0)) / h
    err_one_sided = abs(one_sided - true_derivative)

    # central difference: [f(x+h) - f(x-h)] / (2h)
    center = (f(1.0 + h) - f(1.0 - h)) / (2 * h)
    err_center = abs(center - true_derivative)

    if err_center > 0:
        improvement = err_one_sided / err_center
    else:
        improvement = float('inf')

    print(f"1e{h_power:2d}  {err_one_sided:16.2e}  {err_center:16.2e}  {improvement:12.1f}x")

print("\nWhat to notice:")
print("- as h shrinks, the one-sided error falls linearly (slope -1)")
print("- as h shrinks, the central error falls quadratically (slope -2)")
print("- the central error bottoms out around 1e-11 near h=1e-5; for even smaller h, floating-point rounding takes over and the error grows again")
print("- that's why the code experiments use h=1e-5 or 1e-7: a balance between accuracy and numerical stability")

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('approx. slope =', np.round(slopes, 3))
print('theoretical slope 2x =', 2 * xs)


## Code experiment: walk through different positions and watch the slope change

Below we walk through a row of `x` values for the warm-up function `f(x) = x² + 2x`, printing the function value and the numerical slope side by side, and confirm that at every point the slope matches the analytical derivative `f'(x) = 2x + 2`.

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ Implement `composite_derivative(x)` — the derivative of y=sin(x²)

**Line of reasoning**:
1. Split the layers: outer f(u)=sin(u), inner g(x)=x²; the chain rule gives dy/dx = f'(g(x)) · g'(x)
2. Differentiate each: f'(u)=cos(u), and substituting u=x² gives cos(x²); g'(x)=2x
3. Multiply: the result is cos(x²)·2x, and NumPy broadcasting automatically makes it correct for array inputs too

**Reference input and output**: at x=√(π/2), x²=π/2, so dy/dx=cos(π/2)·2x=0·√(2π)=0 (sin(x²) reaches an extremum at this point, so the derivative is zero)

> 💡 Need a hint? Once you've finished the exercise, you can compare with the reference implementation in the `solutions/` folder.


## An aside: why not just use the formula dy/dx = cos(x²)·2x directly?

You might ask: since we've already derived the answer dy/dx = cos(x²)·2x, why bother splitting the computation into a "forward" and a "backward" part?

**In this 2-layer example, both approaches give the same result.** But once the function gets complicated (like the dozens of layers of a neural network):

| Approach | Complexity | When to use it |
|---|---|---|
| **Direct approach** | derive the complete formula layer by layer, by hand (or with a symbolic-algebra tool) | simple functions (≤ 3 layers); theoretical derivations |
| **Forward–backward approach** | cache the local derivative of each layer, then multiply them in reverse order | complicated functions (dozens of layers); **the foundation of automatic differentiation (autograd)** |

In deep learning, a neural network often has 50+ layers, each with millions of parameters. If you wanted the derivative with respect to every parameter, the direct approach would produce one gigantic symbolic expression (pages and pages of it).

**But the forward–backward approach only needs:**
1. Forward: compute each layer in order, recording that layer's derivative as you go (a few lines of code)
2. Backward: starting from the loss at the last layer, multiply these derivatives together going backward (a few lines of code)

This idea is the heart of the `autograd` engine in Month 2 — the machine does the "split and multiply" for you, and we never hand-write a single derivative formula.

The `composite_derivative` below is the smallest instance of this idea: instead of writing the full formula cos(x²)·2x, we explicitly split out "the inner 2x" and "the outer cos(u)", so the code clearly says what it means: "differentiate layer by layer, then multiply".

---

### Before you write code, fill in the variable table

Before writing `composite_derivative`, pin down three things:
- Input: `x` (a scalar or array, in radians)
- Key step: first compute the inner `g(x) = x²`, then multiply the outer derivative `cos(x²)` by the inner derivative `2x`
- Return: an array of derivatives with the same shape as `x`, ready to compare directly with the numerical difference

In [ ]:
def composite_derivative(x):
    # ✏️ TODO: the derivative of sin(x^2) with respect to x
    # Hint: outer f(u)=sin(u) → f'(u)=cos(u); inner g(x)=x² → g'(x)=2x
    # Chain rule: dy/dx = cos(x²) · 2x
    raise NotImplementedError("TODO: implement composite_derivative — return cos(x**2) * 2*x")


In [ ]:
x = 1.3
numeric = (np.sin((x+1e-5)**2) - np.sin((x-1e-5)**2)) / (2e-5)
try:
    result = composite_derivative(x)
    assert abs(result - numeric) < 1e-6, (
        f"error {abs(result - numeric):.2e} is too large (should be < 1e-6)"
    )
    print("✅ Passed: you've worked through the chain rule by hand once = one minimal backpropagation.")
except (NotImplementedError, TypeError):
    print("⚠️  composite_derivative isn't implemented yet; finish the TODO first, then run this cell.")


In [ ]:
# Aurora connection: the autograd engine in Month 2 (Value.backward in L54) automates this lesson's chain of multiplications.
# Simulate one minimal "backward" by hand: the computation graph of y = sin(x²) has just two edges, each caching its local derivative.
import numpy as np

x = 1.3
u = x**2                  # forward: inner node g(x)
y = np.sin(u)             # forward: outer node f(u)
local_du_dx = 2 * x       # local derivative of edge x→u (can be cached during the forward pass)
local_dy_du = np.cos(u)   # local derivative of edge u→y (can be cached during the forward pass)
grad = local_dy_du * local_du_dx   # backward: multiply along the edges of the computation graph
print(f'dy/dx = {grad:.6f} (matches the chain rule by hand, cos(x²)·2x)')
assert abs(grad - np.cos(x**2) * 2 * x) < 1e-12
print('✅ These few lines are the tiniest prototype of autograd backward — L54 wraps them up into a Value class')

## What does a computation graph look like? Writing it out explicitly in code

Look at the code above — it's really building a "computation graph". Let's write that graph out explicitly as a data structure:

In [ ]:
import numpy as np

# The computation graph of y = sin(x²), represented as a dictionary
x = 1.3
computation_graph = {
    "nodes": {
        "input_x": {"value": x, "type": "input"},
        "u": {"value": x**2, "type": "intermediate"},
        "output_y": {"value": np.sin(x**2), "type": "output"},
    },
    "edges": [
        {
            "from": "input_x",
            "to": "u",
            "operation": "lambda x: x**2",
            "local_gradient": 2*x,  # du/dx at this x
        },
        {
            "from": "u",
            "to": "output_y",
            "operation": "lambda u: np.sin(u)",
            "local_gradient": np.cos(x**2),  # dy/du at this u
        }
    ]
}

print("Nodes of the computation graph:")
for node_name, node_info in computation_graph["nodes"].items():
    print(f"  {node_name}: value={node_info['value']:.6f}, type={node_info['type']}")

print("\nEdges of the computation graph (each edge stores one local derivative):")
for i, edge in enumerate(computation_graph["edges"]):
    print(f"  edge {i}: {edge['from']} -> {edge['to']}")
    print(f"      operation: {edge['operation']}")
    print(f"      local derivative: {edge['local_gradient']:.6f}")

print("\nBackpropagation: passing the gradient from the output layer back to the input layer")
print("  1. Start: the derivative of the output y is 1 (its derivative with respect to itself)")
print("  2. Go backward along the edges: dy/dx = dy/du * du/dx")
dy_du = computation_graph["edges"][1]["local_gradient"]
du_dx = computation_graph["edges"][0]["local_gradient"]
dy_dx = dy_du * du_dx
print(f"           = {dy_du:.6f} * {du_dx:.6f} = {dy_dx:.6f}")
print("\n💡 This process is backpropagation — multiplying the gradient backward along the edges of the computation graph.\n   Month 2's autograd will automate both building this dictionary and passing the gradients.")

## Parameter experiment: the chain rule for a 3-layer composite

Build the 3-layer composite h(x)=sin(exp(x²)):
- Innermost layer u=x², derivative du/dx=2x
- Middle layer v=exp(u), derivative dv/du=exp(u)=exp(x²)
- Outer layer y=sin(v), derivative dy/dv=cos(v)=cos(exp(x²))

The chain rule gives: dh/dx = cos(exp(x²)) · exp(x²) · 2x

Using a numerical difference with step size `h=1e-7`, compute the numerical derivative once at each of x=[0, 1, -1] and compare with the hand-derived value above; the error should be within 1e-9. This confirms that the chain rule extends directly to nesting of any depth — which is exactly what an autograd computation graph does.

## Extending the chain rule to 3 layers: h(x) = sin(exp(x²))

When a composite function has 3 layers, how does the chain rule extend? The rule doesn't change — **multiply your way in, one layer at a time, from the outside**.

Let's make each step explicit in a table:

| Step | Function | Variable | Derivative formula | In our example |
|---|---|---|---|---|
| **1. Outermost layer (start from the output)** | $y = \sin(v)$ | $v$ is an intermediate value | $\frac{dy}{dv} = \cos(v)$ | $\cos(\exp(x^2))$ |
| **2. Middle layer** | $v = \exp(u)$ | $u$ is a deeper intermediate value | $\frac{dv}{du} = \exp(u)$ | $\exp(x^2)$ |
| **3. Innermost layer (reach the input)** | $u = x^2$ | $x$ is the final input | $\frac{du}{dx} = 2x$ | $2x$ |
| **4. Chain product** | — | — | $\frac{dh}{dx} = \frac{dy}{dv} \cdot \frac{dv}{du} \cdot \frac{du}{dx}$ | $\cos(\exp(x^2)) \cdot \exp(x^2) \cdot 2x$ |

**Remember the order**: $\frac{dh}{dx} = [\text{outermost derivative}] \times [\text{middle derivative}] \times [\text{innermost derivative}]$

**Intuition**: a tiny change in x is first "amplified" by a factor of 2x by the derivative of the innermost layer (x²); that change is then amplified by a factor of exp(x²) by the derivative of the middle layer (exp), and finally by a factor of cos(...) by the derivative of the outer layer (sin). Multiply the three amplification factors together and you have the total amplification.

---

In [ ]:
# Parameter experiment: checking the chain rule for h(x) = sin(exp(x²))
# dh/dx = cos(exp(x²)) · exp(x²) · 2x
def h_analytic(x):
    return np.cos(np.exp(x**2)) * np.exp(x**2) * 2 * x

print(f"{'x':>6}  {'analytical':>14}  {'numerical':>14}  {'error':>10}")
print('-' * 50)
for x in [0.0, 1.0, -1.0]:
    analytic = h_analytic(x)
    numerical = (np.sin(np.exp((x+1e-7)**2)) - np.sin(np.exp((x-1e-7)**2))) / 2e-7
    err = abs(analytic - numerical)
    print(f"{x:>6.1f}  {analytic:>14.8f}  {numerical:>14.8f}  {err:>10.2e}")
    assert err < 1e-9, f'x={x}: error {err} too large'
print('✅ The chain rule for sin(exp(x²)) checks out at all three points')


### Extra: the derivative of the exponential function e^x

In the 3-layer chain rule above, the middle layer is $v = \exp(u) = e^u$ ($e \approx 2.718$ is a mathematical constant).

**You may not have met this in high school, so here's a quick top-up:**

The derivative of the exponential function $y = e^x$ is itself:

$$\frac{d}{dx} e^x = e^x$$

**Why?** That's the definition of $e$ — it's the unique base for which "the function's derivative equals the function itself". With $2^x$ or $10^x$, the derivative would be different (off by a constant factor).

In our example, when the middle layer's derivative is $\frac{dv}{du} = e^u$, its value at $u=x^2$ is just $e^{x^2}$.

**Remembering this result is enough** (the full derivation needs limits, which high school doesn't require).

---

In [ ]:
# Going further: the 4-layer composite g(x) = tanh(sin(x²+1))
# dg/dx = (1 - tanh²(sin(x²+1))) · cos(x²+1) · 2x
def g_analytic(x):
    u = x**2 + 1
    v = np.sin(u)
    w = np.tanh(v)
    return (1 - w**2) * np.cos(u) * 2 * x

x = 0.8
analytic = g_analytic(x)
numerical = (np.tanh(np.sin((x+1e-7)**2+1)) - np.tanh(np.sin((x-1e-7)**2+1))) / 2e-7
err = abs(analytic - numerical)
print(f'x={x}: analytical derivative={analytic:.8f}, numerical difference={numerical:.8f}, error={err:.2e}')
assert err < 1e-5
print('✅ The chain rule for the 4-layer composite checks out')
print('  This is exactly what autograd does — only the parameter space has millions of dimensions and the chain is deeper')


### Extra: the derivative of the hyperbolic tangent tanh

The 4-layer composite function used $y = \tanh(x)$ and its derivative $\frac{dy}{dx} = 1 - \tanh^2(x)$. This is another function you won't have met in high school.

**What is tanh?** It's short for hyperbolic tangent. Its definition is:

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

The definition looks complicated, but it has nice properties:
- its output lies between -1 and 1 (like a smooth S-shaped curve)
- its derivative formula is neat: $\frac{d}{dx} \tanh(x) = 1 - \tanh^2(x)$

**Why 1 - tanh²(x)?** You can derive it from the definition above (using the quotient rule and the derivative of e^x), but the working is fairly long. Here, you only need to remember the formula.

In neural networks, $\tanh$ is often used as an **activation function** (coming in Month 2) — it trains faster than sigmoid.

**What it's for in this lesson**: it's a check example for a 4-layer composite function, meant to show that the chain rule works at any level of complexity. You don't need the full derivation of tanh; just trust the derivative formula, and use a numerical difference to check that the answer is right.

---

## 🎯 The payoff later

The "outer derivative × inner derivative" chain of multiplications you took apart yourself today is the seed of all differentiation in deep learning. In **L54 (autograd)** it becomes `Value.backward()`, and it comes straight back in **L56 (backpropagation)** — at that point, differentiating an entire neural network is, at heart, multiplying today's chain layer by layer along the computation graph.

## Wrap-up

You can now call `composite_derivative(x)` to get the derivative of `sin(x²)` at any point, with a numerical error usually within the order of `1e-9`. This "outer derivative × inner derivative" structure is the core of `autograd.backward()`: each node caches its local derivative, and these are multiplied along the computation graph toward the input. When we really implement autograd in Month 2, this way of splitting up `composite_derivative` turns directly into the design of the graph's edges and nodes.

Next lesson: **L25** (gradient descent) uses today's derivative as a direction signal and pushes the parameter one step in the negative-gradient direction. Further ahead, **L54** (autograd) turns this chain rule directly into `Value.backward()`, using a computation graph to do the same multiplications automatically.

---
⬇️ **Clearing check**: you've read the wrap-up; now do the whiteboard challenge below before ticking off the self-review.


## ✏️ Whiteboard challenge: the chain rule by hand (target: 8 minutes)

Cover the screen and answer on paper:

**Question 1**: y = sin(x²). Write out the four steps of the chain rule (split the layers, inner derivative, outer derivative, product).
At x=1, dy/dx = ? (work out the number)

**Question 2**: y = (3x+1)⁵, dy/dx = ?
(Hint: outer f(u)=u⁵, inner g(x)=3x+1)

**Question 3**: h(x) = sin(exp(x²)). Write out the chain-rule expansion of dh/dx.
(A 3-layer composite: multiply in one term per layer)

**Question 4**: In backpropagation, what role does the "chain rule" play?
(Explain in one sentence: the path of computation from the final error to the gradient of the parameters in layer i)

When you've finished the derivations, run the cell below to check your answers.

### How to approach the questions

This whiteboard challenge covers all the key ideas of the lesson:
1. **Questions 1–3**: apply the chain rule one by one, with the number of layers growing from 2 to 3
2. **Question 4**: understand how the chain rule relates to backpropagation

As you work, remember:
- Split the layers first (what's the inner function? what's the outer function?)
- Differentiate each separately (what's each layer's derivative?)
- Multiply from the outside in (don't get the order backward)

### Quick review: the power rule

Question 2 uses $\frac{d}{dx} x^n = n \cdot x^{n-1}$. It's a basic high-school formula, and you may need a quick refresher:

- $\frac{d}{dx} x = 1$
- $\frac{d}{dx} x^2 = 2x$
- $\frac{d}{dx} x^3 = 3x^2$
- $\frac{d}{dx} x^5 = 5x^4$ (Question 2 uses this one)

**Derivation sketch** (with the binomial theorem): $(x+h)^n = x^n + nx^{n-1}h + O(h^2)$, so $\frac{(x+h)^n - x^n}{h} = nx^{n-1} + O(h)$, which gives $nx^{n-1}$ as $h \to 0$.

---

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: y=sin(x²); at x=1, dy/dx = cos(1)·2
x1 = 1.0
dy_dx1 = np.cos(x1**2) * 2 * x1
assert np.isclose(dy_dx1, 2 * np.cos(1.0), atol=1e-12)
try:
    cd1 = composite_derivative(x1)
    assert abs(cd1 - dy_dx1) < 1e-7, f"composite_derivative disagrees with the analytical value: {cd1} vs {dy_dx1}"
    print(f"Q1 ✅  dy/dx|_{{x=1}} = cos(1)·2 = {dy_dx1:.6f}, numerical={cd1:.6f}")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  dy/dx|_{{x=1}} = cos(1)·2 = {dy_dx1:.6f}  (composite_derivative still to implement)")

# Q2: y=(3x+1)⁵, dy/dx = 5(3x+1)⁴·3 = 15(3x+1)⁴
f2 = lambda x: (3*x + 1)**5
df2_analytical = lambda x: 15 * (3*x + 1)**4
x2 = 0.5
expected2 = df2_analytical(x2)
numeric2 = (f2(x2 + 1e-7) - f2(x2 - 1e-7)) / (2e-7)
assert np.isclose(numeric2, expected2, rtol=1e-5), f"derivative of (3x+1)⁵ disagrees: {numeric2} vs {expected2}"
print(f"Q2 ✅  d/dx(3x+1)⁵ = 15(3x+1)⁴; at x={x2} it is {expected2:.4f}, numerical={numeric2:.4f}")

# Q3: h(x)=sin(exp(x²)), dh/dx = cos(exp(x²))·exp(x²)·2x
h = lambda x: np.sin(np.exp(x**2))
dh_analytical = lambda x: np.cos(np.exp(x**2)) * np.exp(x**2) * 2*x
x3 = 0.5
expected3 = dh_analytical(x3)
numeric3 = (h(x3 + 1e-7) - h(x3 - 1e-7)) / (2e-7)
assert np.isclose(numeric3, expected3, rtol=1e-5), f"derivative of sin(exp(x²)) disagrees"
print(f"Q3 ✅  dh/dx=cos(e^x²)·e^x²·2x; at x={x3} it is {expected3:.6f}, numerical={numeric3:.6f}")

# Q4: backpropagation is the chain rule carried out in bulk (check the 2-layer chain rule)
# y=f(g(x)), dy/dx=f'(g(x))·g'(x) — check the chain of multiplications
g = lambda x: x**2
f_outer = lambda u: np.sin(u)
dg = lambda x: 2*x
df = lambda u: np.cos(u)
x4 = 1.3
chain = df(g(x4)) * dg(x4)
direct_num = (f_outer(g(x4 + 1e-7)) - f_outer(g(x4 - 1e-7))) / (2e-7)
assert np.isclose(chain, direct_num, rtol=1e-6)
print(f"Q4 ✅  chain product f'(g(x))·g'(x) = {chain:.6f} = numerical difference {direct_num:.6f}")
print("     BP = start from the output-layer error and multiply by each layer's local derivative, layer by layer going back, until you reach the parameter layer")
print("\n🎉 Chain-rule whiteboard challenge passed! You've internalized the maths behind backpropagation.")

In [ ]:
# ✏️ Self-review for this lesson
l24_review = {
    "chain_rule_formula":         None,  # remember dy/dx = f'(g(x))·g'(x)? True/False
    "composite_deriv_done":       None,  # composite_derivative implemented and assertions pass? True/False
    "3_layer_chain":              None,  # can expand the chain product of a 3-layer composite? True/False
    "backprop_connection":        None,  # understand BP = the chain rule carried out in bulk on a computation graph? True/False
    "whiteboard_passed":          None,  # whiteboard challenge done on paper? True/False
}

unfilled = [k for k, v in l24_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l24_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L24 fully cleared! On to L25: gradient descent')

---

→ **Next lesson** · [L25 · Gradient descent](L25_gradient_descent.ipynb)

> Next lesson covers **gradient descent**: walking downhill blindfolded, following the negative gradient to the bottom of the valley — from loss function to weight-update rule.